In [1]:
# Import Libraries

import pandas as pd
from pathlib import Path


In [2]:
# Loads the existing training, validation, and test splits. 
# Feature formulas must be identical across them; learned transformations belong in training-fitted pipelines.
train = pd.read_csv("data/processed/train.csv")
validation = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")


In [3]:
# Creates copies so engineered features can be added without overwriting the original split DataFrames. Split membership is preserved.

train_fe = train.copy()
validation_fe = validation.copy()
test_fe = test.copy()

splits = [train_fe, validation_fe, test_fe]


In [4]:
# convert age to numerical (midpoints)
# Replaces each age band with its midpoint in a new column. This is an approximate age, not the exact patient age, and the mapping does not use outcome information.

age_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95,
}

for df in splits:
    df["age_numeric"] = df["age"].map(age_map)


In [5]:
# calculate total previous healthcare utilization
# Add previous outpatient, emergency, and inpatient encounters.
# This summarizes recorded healthcare use and overlaps with the three retained component variables.

for df in splits:
    df["prior_utilization"] = (
        df["number_outpatient"]
        + df["number_emergency"]
        + df["number_inpatient"]
    )


In [6]:
# Flag previous inpatient encounters
# 1 means at least one previous inpatient encounter; 0 means none.
# This can express the difference between no prior admission and at least one, alongside the original count.

for df in splits:
    df["prior_inpatient"] = (
        df["number_inpatient"] > 0
    ).astype(int)


In [7]:
# List the retained medication columns
# Lists the 17 retained medication-status columns used by the following features. The count represents these columns, not all possible treatments or unique drug ingredients.

med_cols = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "tolazamide",
    "troglitazone",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
]


In [8]:
# Check medication values 
# checks that all medication entries use the expected labels before we count them.
# The saved assertion output confirms expected medication labels. This protects the counting formulas from unexpected values that could silently be treated as non-use.

for df in splits:
    assert df[med_cols].isin(
        ["No", "Steady", "Up", "Down"]
    ).all().all()

print("Medication values are valid.")


Medication values are valid.


In [9]:
# checks that all medication entries use the expected labels before we count them.
# Count Steady, Up, or Down as medication use. This counts medication columns, not unique drug ingredients.
# The feature measures recorded treatment use, not dosage, adherence, or treatment effectiveness.


for df in splits:
    df["num_diabetes_meds"] = (
        df[med_cols]
        .isin(["Steady", "Up", "Down"])
        .sum(axis=1)
    )


In [10]:
# Insulin use
# Flags recorded insulin use regardless of whether its status is steady, increased, or decreased. This feature overlaps with the original insulin category.

for df in splits:
    df["insulin_used"] = (
        df["insulin"]
        .isin(["Steady", "Up", "Down"])
        .astype(int)
    )


In [11]:
# Count diabetes diagnoses
# Count how many of the three diagnosis groups are labeled "Diabetes".
# Counts diabetes labels among three retained diagnosis positions. 
# A count of zero does not mean the patient has no diabetes; diabetes may be recorded outside those three positions.



diag_cols = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
]

for df in splits:
    df["diabetes_diag_count"] = (
        df[diag_cols]
        .eq("Diabetes")
        .sum(axis=1)
    )


In [12]:
# Check new features
# Checks all six new features for missingness and confirms that the outcomes are unchanged. 
# Passing these checks supports implementation consistency, not proof of predictive improvement.

new_features = [
    "age_numeric",
    "prior_utilization",
    "prior_inpatient",
    "num_diabetes_meds",
    "insulin_used",
    "diabetes_diag_count",
]

for df in splits:
    assert df[new_features].notna().all().all()

# Confirm the target has not changed
assert train_fe["readmit_30"].equals(train["readmit_30"])
assert validation_fe["readmit_30"].equals(validation["readmit_30"])
assert test_fe["readmit_30"].equals(test["readmit_30"])

print("Checks passed.")


Checks passed.


In [13]:
# View training examples
# Displays training examples to make the formulas easy to inspect. A small preview is a spot check and should be interpreted alongside the assertions.

train_fe[
    ["age"] + new_features + ["readmit_30"]
].head(10)


Out[13]: 
       age  age_numeric  ...  diabetes_diag_count  readmit_30
0  [60-70)           65  ...                    2           0
1  [70-80)           75  ...                    0           0
2  [80-90)           85  ...                    1           0
3  [70-80)           75  ...                    0           0
4  [60-70)           65  ...                    0           0
5  [50-60)           55  ...                    1           0
6  [30-40)           35  ...                    1           0
7  [70-80)           75  ...                    0           0
8  [40-50)           45  ...                    1           0
9  [10-20)           15  ...                    1           0

[10 rows x 8 columns]


,age,age_numeric,prior_utilization,prior_inpatient,num_diabetes_meds,insulin_used,diabetes_diag_count,readmit_30
0,[60-70),65,0,0,1,1,2,0
1,[70-80),75,0,0,0,0,0,0
2,[80-90),85,1,0,0,0,1,0
3,[70-80),75,0,0,1,1,0,0
4,[60-70),65,0,0,1,0,0,0
5,[50-60),55,0,0,0,0,1,0
6,[30-40),35,0,0,1,1,1,0
7,[70-80),75,3,1,3,1,0,0
8,[40-50),45,0,0,1,1,1,0
9,[10-20),15,0,0,1,1,1,0


In [14]:
# Summarize the training features

train_fe[new_features].describe().T

# The saved training summary shows age midpoint mean 65.47, insulin use 51.00%, and prior inpatient history 11.69%. 
# Prior utilization reaches 42 - for review as to why


Out[14]: 
                       count       mean        std  min   25%   50%   75%   max
age_numeric          48978.0  65.467149  15.933248  5.0  55.0  65.0  75.0  95.0
prior_utilization    48978.0   0.561048   1.426246  0.0   0.0   0.0   1.0  42.0
prior_inpatient      48978.0   0.116869   0.321267  0.0   0.0   0.0   0.0   1.0
num_diabetes_meds    48978.0   1.188697   0.941497  0.0   1.0   1.0   2.0   5.0
insulin_used         48978.0   0.509964   0.499906  0.0   0.0   1.0   1.0   1.0
diabetes_diag_count  48978.0   0.398975   0.501638  0.0   0.0   0.0   1.0   3.0


,count,mean,std,min,25%,50%,75%,max
age_numeric,48978.0,65.467149,15.933248,5.0,55.0,65.0,75.0,95.0
prior_utilization,48978.0,0.561048,1.426246,0.0,0.0,0.0,1.0,42.0
prior_inpatient,48978.0,0.116869,0.321267,0.0,0.0,0.0,0.0,1.0
num_diabetes_meds,48978.0,1.188697,0.941497,0.0,1.0,1.0,2.0,5.0
insulin_used,48978.0,0.509964,0.499906,0.0,0.0,1.0,1.0,1.0
diabetes_diag_count,48978.0,0.398975,0.501638,0.0,0.0,0.0,1.0,3.0


In [15]:
# Save the engineered datasets
# Saves train_fe.csv, val_fe.csv, and test_fe.csv for modeling. 
# Feature engineering preserves the target and applies the same six fixed formulas to every split.

Path("data/processed").mkdir(parents=True, exist_ok=True)

train_fe.to_csv("data/processed/train_fe.csv", index=False)
validation_fe.to_csv("data/processed/val_fe.csv", index=False)
test_fe.to_csv("data/processed/test_fe.csv", index=False)

print("Saved train_fe.csv, val_fe.csv, and test_fe.csv")


Saved train_fe.csv, val_fe.csv, and test_fe.csv
